# Treino da LSTM — quantos pods serão necessários daqui a 10 minutos

Notebook para o **Google Colab**. Usa dados **reais** (NASA 1995 ou Copa do Mundo 1998),
agregados por minuto, como em Dang-Quang & Yoo (2021) e Guruge & Priyadarshana (2025).

**O que acontece, em ordem**

1. Baixa o código do projeto do GitHub (os mesmos scripts documentados em `docs/pre_processamento/`)
2. Carrega a série de requisições por minuto
3. Monta os exemplos: **10 minutos de histórico → os próximos 10 minutos**
4. Treina **LSTM, Bi-LSTM e GRU**, as três redes mais usadas nos artigos
5. Compara com **modelos simples** (se um modelo simples empatar, ele vence)
6. Mede o erro **minuto a minuto** do horizonte (t+1 … t+10)
7. Converte a previsão em **número de pods** e compara com o reativo (tipo HPA)
8. Salva tudo (modelos, métricas, previsões, versões) num `.zip` para baixar

**Como usar:** ajuste a célula de configuração e clique em *Ambiente de execução → Executar tudo*.
GPU é opcional: *Ambiente de execução → Alterar o tipo de ambiente → T4 GPU*.

## 1. Configuração — o único lugar que você precisa mexer

In [ ]:
BASE = "nasa"            # "nasa" ou "fifa"
JANELA = 10              # minutos de histórico que a rede vê (igual ao artigo)
HORIZONTE = 10           # quantos minutos à frente ela prevê: t+1, t+2, ..., t+10
FRACAO_TREINO = 0.70     # 70% treino / 30% teste, NA ORDEM DO TEMPO
FRACAO_VALIDACAO = 0.10  # últimos 10% do treino: usados para decidir quando parar
USAR_CALENDARIO = False  # True = a rede também recebe hora do dia e dia da semana
REDES = ["LSTM", "Bi-LSTM", "GRU"]
UNIDADES = 30            # neurônios da camada recorrente (igual ao artigo)
EPOCAS, LOTE, PACIENCIA = 100, 64, 10
PODS_NO_PICO = 20        # capacidade de 1 pod = pico do treino / 20  (ver seção 7)
SEMENTE = 42
REPO = "https://github.com/pedro-assuncao10/projeto_autoscaling_lstm_green_it.git"

## 2. Ambiente: baixa o projeto e registra as versões

In [ ]:
import os, sys, json, math, time, platform, subprocess
from datetime import datetime, timezone

RAIZ = "/content/projeto"

def rodar(cmd, **kw):
    r = subprocess.run(cmd, capture_output=True, text=True, **kw)
    print(r.stdout + r.stderr)
    r.check_returncode()
    return r.stdout

if not os.path.exists(RAIZ):
    rodar(["git", "clone", "--depth", "1", REPO, RAIZ])
os.chdir(f"{RAIZ}/reproducao")
sys.path.insert(0, "src")

import numpy as np, pandas as pd, tensorflow as tf
import matplotlib.pyplot as plt
tf.keras.utils.set_random_seed(SEMENTE)

AMBIENTE = {
    "commit_do_projeto": rodar(["git", "rev-parse", "--short", "HEAD"]).strip(),
    "python": platform.python_version(),
    "numpy": np.__version__,
    "pandas": pd.__version__,
    "tensorflow": tf.__version__,
    "gpu": [g.name for g in tf.config.list_physical_devices("GPU")] or "nenhuma",
    "executado_em_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
}
AMBIENTE

## 3. Dados

- **NASA:** se o arquivo não existir, é gerado aqui mesmo com os scripts do projeto
  (baixa os logs originais e conta as requisições por minuto, cerca de 1 minuto).
- **Copa 1998:** são vários GB de logs. Gere `fifa_1min.csv` no seu computador e
  **envie o arquivo** quando a célula pedir (está em `reproducao/dados/`).

Se a base tiver período sem medição (`valido = 0`, ex.: furacão Erin na NASA), usamos o
**maior trecho contínuo** sem falhas, como em `src/dados.py`.

In [ ]:
from dados import maior_trecho_valido

ARQ = f"dados/{BASE}_1min.csv"
if not os.path.exists(ARQ):
    if BASE == "nasa":
        rodar([sys.executable, "src/baixar_tracos.py", "nasa"])
        rodar([sys.executable, "src/preparar_traco.py", "nasa", ARQ])
    else:
        from google.colab import files
        print("Envie o arquivo fifa_1min.csv (gerado no seu computador, em reproducao/dados/):")
        enviado = files.upload()
        os.makedirs("dados", exist_ok=True)
        os.replace(next(iter(enviado)), ARQ)

df = pd.read_csv(ARQ, parse_dates=["timestamp"])
if "valido" in df and not df["valido"].all():
    a, b = maior_trecho_valido(df["valido"].to_numpy(bool))
    print(f"Período sem medição encontrado: usando o maior trecho contínuo ({b - a:,} de {len(df):,} minutos)")
    df = df.iloc[a:b].reset_index(drop=True)

carga = df["requests_per_minute"].to_numpy(float)
print(f"{BASE}: {len(df):,} minutos ({len(df)/1440:.1f} dias) | "
      f"{df.timestamp.iloc[0]:%d/%m/%Y %H:%M} a {df.timestamp.iloc[-1]:%d/%m/%Y %H:%M} UTC")
print(f"req/min: mín={carga.min():.0f} | média={carga.mean():.0f} | máx={carga.max():.0f}")

fig, ax = plt.subplots(2, 1, figsize=(14, 6))
ax[0].plot(df.timestamp, carga, lw=0.3); ax[0].set_title(f"{BASE}: série completa"); ax[0].set_ylabel("req/min")
d3 = df.timestamp < df.timestamp.iloc[0] + pd.Timedelta(days=3)
ax[1].plot(df.timestamp[d3], carga[d3], lw=0.7); ax[1].set_title("três primeiros dias"); ax[1].set_ylabel("req/min")
plt.tight_layout(); plt.show()

## 4. Exemplos de treino: 10 minutos de histórico → próximos 10 minutos

Cada exemplo é uma "foto" da série:

```
entrada  (o que a rede vê)  : carga dos minutos t-9 ... t
resposta (o que ela prevê)  : carga dos minutos t+1 ... t+10
```

Cuidados para não "colar" do futuro:

- **Divisão pela ordem do tempo:** os primeiros 70% para treino, o resto para teste.
  Nada é embaralhado antes da divisão.
- **Normalização (0 a 1) calculada só com o treino.**
- **Exemplos na fronteira são descartados:** nenhum exemplo de treino tem resposta
  dentro do período de teste.

In [ ]:
corte = int(len(carga) * FRACAO_TREINO)
vmin, vmax = carga[:corte].min(), carga[:corte].max()
normalizar = lambda v: (v - vmin) / (vmax - vmin)
desnormalizar = lambda v: v * (vmax - vmin) + vmin

entradas = [normalizar(carga)]
if USAR_CALENDARIO:                      # seno/cosseno: 23h fica "perto" de 0h
    hora = df.timestamp.dt.hour + df.timestamp.dt.minute / 60
    dia = df.timestamp.dt.dayofweek
    entradas += [np.sin(2 * np.pi * hora / 24), np.cos(2 * np.pi * hora / 24),
                 np.sin(2 * np.pi * dia / 7),   np.cos(2 * np.pi * dia / 7)]
F = np.stack(entradas, axis=1).astype("float32")             # (minutos, n_entradas)

from numpy.lib.stride_tricks import sliding_window_view as janelas
n = len(carga) - JANELA - HORIZONTE + 1
X = janelas(F, JANELA, axis=0)[:n].transpose(0, 2, 1)       # (n, JANELA, n_entradas)
Y = janelas(F[:, 0], HORIZONTE)[JANELA:JANELA + n]           # (n, HORIZONTE)
inicio_alvo = np.arange(n) + JANELA                          # índice do minuto t+1 de cada exemplo

treino = inicio_alvo + HORIZONTE <= corte                    # resposta inteira antes do corte
teste = inicio_alvo >= corte                                 # resposta inteira depois do corte
idx_tr = np.where(treino)[0]
n_val = int(len(idx_tr) * FRACAO_VALIDACAO)
idx_val, idx_tr = idx_tr[-n_val:], idx_tr[:-n_val]
idx_te = np.where(teste)[0]

X_tr, Y_tr = X[idx_tr], Y[idx_tr]
X_val, Y_val = X[idx_val], Y[idx_val]
X_te = X[idx_te]
real_te = janelas(carga, HORIZONTE)[JANELA:JANELA + n][idx_te]   # resposta real, em req/min
ts_te = df.timestamp.to_numpy()[inicio_alvo[idx_te]]

print(f"entradas por minuto: {X.shape[2]}  ({'carga + calendário' if USAR_CALENDARIO else 'só a carga'})")
print(f"treino: {len(idx_tr):,} | validação: {len(idx_val):,} | teste: {len(idx_te):,} exemplos")

## 5. As redes

As três têm a mesma estrutura, só muda a camada recorrente:

```
entrada (10 minutos) → camada recorrente (30 neurônios) → camada densa → 10 valores (t+1 … t+10)
```

| Rede | O que é |
|---|---|
| **LSTM** | lê os 10 minutos em ordem e guarda uma "memória" do que importa |
| **Bi-LSTM** | duas LSTMs: uma lê do mais antigo ao mais novo, outra ao contrário (o artigo de referência) |
| **GRU** | versão simplificada da LSTM, com menos parâmetros: treina mais rápido |

O treino para sozinho quando o erro na validação para de melhorar por `PACIENCIA`
épocas, e fica com a melhor versão encontrada.

In [ ]:
from tensorflow.keras import layers, models, callbacks

def construir(tipo):
    ent = layers.Input((JANELA, X.shape[2]))
    if tipo == "LSTM":
        h = layers.LSTM(UNIDADES)(ent)
    elif tipo == "Bi-LSTM":
        h = layers.Bidirectional(layers.LSTM(UNIDADES))(ent)
    elif tipo == "GRU":
        h = layers.GRU(UNIDADES)(ent)
    m = models.Model(ent, layers.Dense(HORIZONTE)(h), name=tipo.replace("-", ""))
    m.compile(optimizer="adam", loss="mse")
    return m

previsoes, modelos, treinos = {}, {}, {}
for tipo in REDES:
    print(f"\n=== {tipo}")
    tf.keras.utils.set_random_seed(SEMENTE)
    m = construir(tipo)
    t0 = time.perf_counter()
    hist = m.fit(X_tr, Y_tr, validation_data=(X_val, Y_val), epochs=EPOCAS, batch_size=LOTE,
                 verbose=2, callbacks=[callbacks.EarlyStopping(patience=PACIENCIA, restore_best_weights=True)])
    t_treino = time.perf_counter() - t0
    t0 = time.perf_counter()
    previsoes[tipo] = desnormalizar(m.predict(X_te, verbose=0))
    t_pred_ms = (time.perf_counter() - t0) * 1000 / len(X_te)
    modelos[tipo] = m
    treinos[tipo] = {"parametros": int(m.count_params()), "epocas": len(hist.history["loss"]),
                     "melhor_val_loss": float(min(hist.history["val_loss"])),
                     "tempo_treino_s": round(t_treino, 1), "tempo_predicao_ms_por_exemplo": round(t_pred_ms, 4),
                     "historico": hist.history}
    print(f"{tipo}: {treinos[tipo]['parametros']:,} parâmetros | {treinos[tipo]['epocas']} épocas | "
          f"{t_treino:.0f} s de treino")

plt.figure(figsize=(10, 4))
for tipo, t in treinos.items():
    plt.plot(t["historico"]["val_loss"], label=f"{tipo} (validação)")
plt.yscale("log"); plt.xlabel("época"); plt.ylabel("erro (MSE, normalizado)"); plt.legend()
plt.title("Curva de aprendizado"); plt.show()

## 6. Modelos simples (baselines)

Uma rede neural só se justifica se vencer o que é simples e barato:

| Baseline | Previsão para os próximos 10 minutos |
|---|---|
| **Persistência** | repete o valor de agora. **É o que um autoscaler reativo (HPA) faz**: decide pela carga atual |
| **Média móvel** | média dos últimos 10 minutos |
| **Mesmo horário de ontem** | a carga de 24 h antes de cada minuto previsto |

In [ ]:
ultimo = desnormalizar(X_te[:, -1, 0])
previsoes["Persistência"] = np.repeat(ultimo[:, None], HORIZONTE, axis=1)
previsoes["Média móvel (10 min)"] = np.repeat(desnormalizar(X_te[:, :, 0]).mean(axis=1)[:, None], HORIZONTE, axis=1)
idx_ontem = inicio_alvo[idx_te][:, None] + np.arange(HORIZONTE)[None, :] - 1440
previsoes["Mesmo horário de ontem"] = carga[idx_ontem]

## 7. Erro da previsão, minuto a minuto do horizonte

| Métrica | Leitura |
|---|---|
| **RMSE** | erro típico, em req/min; pune mais os erros grandes (os picos) |
| **MAE** | erro médio, em req/min |
| **WAPE** | erro total ÷ carga total, em %. Mais confiável que o MAPE quando há minutos com carga quase zero |
| **MAPE** | erro percentual médio. Usado nos artigos, mas **explode** com carga perto de zero (madrugada da NASA) |
| **R²** | 1 = perfeito; 0 = tão bom quanto prever sempre a média |

In [ ]:
def metricas(real, prev):
    e = real - prev
    nz = real > 0
    return {"RMSE": float(np.sqrt(np.mean(e ** 2))),
            "MAE": float(np.mean(np.abs(e))),
            "WAPE_%": float(np.sum(np.abs(e)) / np.sum(np.abs(real)) * 100),
            "MAPE_%": float(np.mean(np.abs(e[nz] / real[nz])) * 100),
            "R2": float(1 - np.sum(e ** 2) / np.sum((real - real.mean()) ** 2))}

linhas = []
for nome, p in previsoes.items():
    for h in range(HORIZONTE):
        linhas.append({"modelo": nome, "minutos_a_frente": h + 1, **metricas(real_te[:, h], p[:, h])})
por_horizonte = pd.DataFrame(linhas)

print(f"Erro ao prever {HORIZONTE} minutos à frente (o que decide os pods):")
display(por_horizonte[por_horizonte.minutos_a_frente == HORIZONTE]
        .drop(columns="minutos_a_frente").set_index("modelo").sort_values("RMSE").round(3))

fig, ax = plt.subplots(1, 2, figsize=(14, 4))
for nome, g in por_horizonte.groupby("modelo"):
    ax[0].plot(g.minutos_a_frente, g.RMSE, marker="o", label=nome)
    ax[1].plot(g.minutos_a_frente, g["WAPE_%"], marker="o", label=nome)
ax[0].set_title("RMSE por minuto à frente"); ax[0].set_xlabel("minutos à frente"); ax[0].set_ylabel("req/min")
ax[1].set_title("WAPE por minuto à frente"); ax[1].set_xlabel("minutos à frente"); ax[1].set_ylabel("%")
ax[0].legend(); plt.tight_layout(); plt.show()

## 8. Da previsão para o número de pods

**Capacidade de um pod.** As bases não dizem quantas requisições um pod atende, então
fixamos uma regra simples e registrada: a capacidade é tal que **o pico do treino exige
`PODS_NO_PICO` pods**. Assim o número de pods fica numa faixa realista para qualquer base.

```
pods(carga) = arredondar para cima( carga / capacidade )      (mínimo 1)
```

Para cada minuto do teste comparamos os pods **previstos para daqui a 10 minutos** com os
pods que a carga real **de fato exigiu** naquele minuto.

- **Faltou pod:** a aplicação ficaria sobrecarregada (violação de SLA)
- **Sobrou pod:** pod ligado à toa (**desperdício — é aqui que entra a energia depois**)

In [ ]:
CAPACIDADE = carga[:corte].max() / PODS_NO_PICO
pods = lambda c: np.maximum(1, np.ceil(np.asarray(c) / CAPACIDADE)).astype(int)
print(f"capacidade de 1 pod = {CAPACIDADE:.1f} req/min")

pods_reais = pods(real_te[:, -1])
tabela_pods = {}
for nome, p in previsoes.items():
    pp = pods(p[:, -1])
    dif = pp - pods_reais
    tabela_pods[nome] = {"acertou_%": np.mean(dif == 0) * 100,
                         "faltou_%_do_tempo": np.mean(dif < 0) * 100,
                         "sobrou_%_do_tempo": np.mean(dif > 0) * 100,
                         "pods_faltando_por_minuto": float(np.mean(np.maximum(-dif, 0))),
                         "pods_sobrando_por_minuto": float(np.mean(np.maximum(dif, 0)))}
tabela_pods = pd.DataFrame(tabela_pods).T.sort_values("faltou_%_do_tempo")
print("'Persistência' = decisão reativa, tipo HPA: usa a carga de agora para os próximos 10 minutos")
display(tabela_pods.round(2))

In [ ]:
dias = min(2, len(ts_te) // 1440)
fatia = slice(0, dias * 1440)
fig, ax = plt.subplots(2, 1, figsize=(14, 7), sharex=True)
ax[0].plot(ts_te[fatia], real_te[fatia, -1], color="black", lw=1, label="carga real")
for nome in REDES + ["Persistência"]:
    ax[0].plot(ts_te[fatia], previsoes[nome][fatia, -1], lw=0.8, alpha=0.8, label=nome)
ax[0].set_ylabel("req/min"); ax[0].legend(); ax[0].set_title(f"Previsão para {HORIZONTE} minutos à frente — {dias} dias do teste")
ax[1].step(ts_te[fatia], pods_reais[fatia], color="black", where="post", label="pods necessários")
for nome in REDES + ["Persistência"]:
    ax[1].step(ts_te[fatia], pods(previsoes[nome][fatia, -1]), where="post", alpha=0.7, label=nome)
ax[1].set_ylabel("pods"); ax[1].legend(); ax[1].set_title("Pods")
plt.tight_layout(); plt.show()

## 9. Salvar tudo

Gera uma pasta com modelos, métricas, previsões e a configuração exata usada, compacta
e baixa para o seu computador. Guarde em `reproducao/resultados/` e registre a execução
em `docs/treino/`.

In [ ]:
carimbo = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M")
SAIDA = f"resultados/{BASE}_h{HORIZONTE}_{carimbo}"
os.makedirs(SAIDA, exist_ok=True)

config = {k: globals()[k] for k in ["BASE", "JANELA", "HORIZONTE", "FRACAO_TREINO", "FRACAO_VALIDACAO",
          "USAR_CALENDARIO", "REDES", "UNIDADES", "EPOCAS", "LOTE", "PACIENCIA", "PODS_NO_PICO", "SEMENTE"]}
config["capacidade_pod_req_min"] = CAPACIDADE
config["normalizacao"] = {"min": float(vmin), "max": float(vmax)}
config["periodo"] = {"inicio": str(df.timestamp.iloc[0]), "fim": str(df.timestamp.iloc[-1]),
                     "corte_treino_teste": str(df.timestamp.iloc[corte])}
with open(f"{SAIDA}/execucao.json", "w", encoding="utf-8") as f:
    json.dump({"ambiente": AMBIENTE, "configuracao": config, "treino": treinos}, f, indent=2, ensure_ascii=False)

por_horizonte.to_csv(f"{SAIDA}/metricas_por_horizonte.csv", index=False)
tabela_pods.to_csv(f"{SAIDA}/pods.csv")
prev_df = pd.DataFrame({"timestamp_t+1": ts_te, f"real_t+{HORIZONTE}": real_te[:, -1]})
for nome, p in previsoes.items():
    prev_df[f"{nome}_t+{HORIZONTE}"] = p[:, -1]
prev_df.to_csv(f"{SAIDA}/previsoes_teste.csv", index=False)
for tipo, m in modelos.items():
    m.save(f"{SAIDA}/{tipo}.keras")

import shutil
zip_ = shutil.make_archive(SAIDA, "zip", SAIDA)
print("salvo:", zip_)
try:
    from google.colab import files
    files.download(zip_)
except ImportError:
    pass